**Evaluación y Validación Robusta (N=85)**

Imports principales de librerías y backend de gráficos

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

Imports de validación y métricas

In [2]:
from sklearn.model_selection import (
    RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
)
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, make_scorer,
    accuracy_score, recall_score, precision_score, ConfusionMatrixDisplay
)

Variables de configuración y directorios

In [3]:
CARPETA_RESULTADOS = "results"
CARPETA_FIGURAS = "fig"
CARPETA_MODELOS = "models"
SEMILLA = 42

Creación de carpetas del proyecto

In [4]:
os.makedirs(CARPETA_FIGURAS, exist_ok=True)
os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
os.makedirs(CARPETA_MODELOS, exist_ok=True)

Carga del dataset completo

In [5]:
# 1. Carga del dataset COMPLETO (85 pacientes, sin holdout)
X = pd.read_csv(f"{CARPETA_RESULTADOS}/X_full.csv")
y = pd.read_csv(f"{CARPETA_RESULTADOS}/y_full.csv").squeeze()
 
print(f"Dimensiones de evaluación -> X: {X.shape} | y: {y.shape}  (85 pacientes, sin holdout)")
 
peso_clase_positiva = (y == 0).sum() / (y == 1).sum()

Dimensiones de evaluación -> X: (85, 17) | y: (85,)  (85 pacientes, sin holdout)


Definición de los pipelines por modelo

In [6]:
# 2. Definición de Pipelines por Modelo
pipelines = {
    'XGBoost': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', xgb.XGBClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            scale_pos_weight=peso_clase_positiva,
            random_state=SEMILLA,
            eval_metric='logloss'
        ))
    ]),
    'Regresión Logística': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            penalty='elasticnet',
            l1_ratio=0.5,
            solver='saga',
            class_weight='balanced',
            max_iter=5000,
            random_state=SEMILLA
        ))
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', RandomForestClassifier(
            n_estimators=200,
            max_depth=4,
            class_weight='balanced',
            random_state=SEMILLA
        ))
    ])
}

Inclusión de TabPFN

In [7]:
try:
    from tabpfn import TabPFNClassifier
    pipelines['TabPFN'] = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('classifier', TabPFNClassifier(random_state=SEMILLA))
    ])
    print("\n[OK] TabPFN disponible: se incluye en la evaluación final.")
except Exception as e:
    print(f"\n[AVISO] No pudo incluirse TabPFN en la evaluación final (omitiendo): {e}")


[OK] TabPFN disponible: se incluye en la evaluación final.


 **A) RENDIMIENTO ROBUSTO POR CV (10-fold x 5 repeticiones)**

Ejecución de la validación cruzada robusta

In [8]:
cv_robusta = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=SEMILLA)
 

print("A) RENDIMIENTO POR CV ROBUSTA (10-fold x 5 repeticiones, N=85)")

 
# Especificidad no tiene scorer directo en sklearn; se calcula como
# recall de la clase negativa usando make_scorer con pos_label=0
especificidad_scorer = make_scorer(recall_score, pos_label=0)
 
# IMPORTANTE (rendimiento): se usa cross_validate con varios scorers a la
# vez en vez de cinco llamadas separadas a cross_val_score. Esto ajusta
# cada uno de los 50 folds UNA sola vez por modelo (no cinco veces), lo
# cual es especialmente importante con TabPFN, que es lento de entrenar:
# evita multiplicar x5 el tiempo de cómputo de la parte más costosa.
scoring_dict = {
    'auc': 'roc_auc',
    'sensibilidad': 'recall',
    'precision': 'precision',
    'exactitud': 'accuracy',
    'especificidad': especificidad_scorer,
}

A) RENDIMIENTO POR CV ROBUSTA (10-fold x 5 repeticiones, N=85)


In [9]:
tabla_metricas_cv = []
for nombre, pipe in pipelines.items():
    print(f"\nEvaluando {nombre}")
    cv_result = cross_validate(pipe, X, y, cv=cv_robusta, scoring=scoring_dict, n_jobs=1)
    scores_auc = cv_result['test_auc']
    scores_sens = cv_result['test_sensibilidad']
    scores_prec = cv_result['test_precision']
    scores_acc = cv_result['test_exactitud']
    scores_esp = cv_result['test_especificidad']
 
    print(f"--- {nombre} ---")
    print(f"AUC-ROC:       {scores_auc.mean():.3f} (+/- {scores_auc.std():.3f})")
    print(f"Sensibilidad:  {scores_sens.mean():.3f} (+/- {scores_sens.std():.3f})")
    print(f"Especificidad: {scores_esp.mean():.3f} (+/- {scores_esp.std():.3f})")
    print(f"Precisión:     {scores_prec.mean():.3f} (+/- {scores_prec.std():.3f})")
    print(f"Exactitud:     {scores_acc.mean():.3f} (+/- {scores_acc.std():.3f})")
 
    tabla_metricas_cv.append({
        'Modelo': nombre,
        'AUC-ROC_media': round(scores_auc.mean(), 3), 'AUC-ROC_std': round(scores_auc.std(), 3),
        'Sensibilidad_media': round(scores_sens.mean(), 3), 'Sensibilidad_std': round(scores_sens.std(), 3),
        'Especificidad_media': round(scores_esp.mean(), 3), 'Especificidad_std': round(scores_esp.std(), 3),
        'Precisión_media': round(scores_prec.mean(), 3), 'Precisión_std': round(scores_prec.std(), 3),
        'Exactitud_media': round(scores_acc.mean(), 3), 'Exactitud_std': round(scores_acc.std(), 3),
    })
 
df_metricas_cv = pd.DataFrame(tabla_metricas_cv)
df_metricas_cv.to_csv(f"{CARPETA_RESULTADOS}/metricas_finales_cv_robusta.csv", index=False)
 
print("\n[OK] Tabla de métricas por CV robusta guardada en "
      f"'{CARPETA_RESULTADOS}/metricas_finales_cv_robusta.csv' -- ESTA es la cifra a citar en la memoria.")


Evaluando XGBoost
--- XGBoost ---
AUC-ROC:       0.820 (+/- 0.143)
Sensibilidad:  0.633 (+/- 0.277)
Especificidad: 0.793 (+/- 0.196)
Precisión:     0.683 (+/- 0.274)
Exactitud:     0.737 (+/- 0.151)

Evaluando Regresión Logística
--- Regresión Logística ---
AUC-ROC:       0.867 (+/- 0.136)
Sensibilidad:  0.820 (+/- 0.202)
Especificidad: 0.790 (+/- 0.190)
Precisión:     0.720 (+/- 0.217)
Exactitud:     0.802 (+/- 0.153)

Evaluando Random Forest


sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


--- Random Forest ---
AUC-ROC:       0.851 (+/- 0.143)
Sensibilidad:  0.607 (+/- 0.272)
Especificidad: 0.839 (+/- 0.186)
Precisión:     0.715 (+/- 0.289)
Exactitud:     0.758 (+/- 0.148)

Evaluando TabPFN


sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


--- TabPFN ---
AUC-ROC:       0.873 (+/- 0.136)
Sensibilidad:  0.693 (+/- 0.265)
Especificidad: 0.825 (+/- 0.186)
Precisión:     0.700 (+/- 0.283)
Exactitud:     0.779 (+/- 0.146)

[OK] Tabla de métricas por CV robusta guardada en 'results/metricas_finales_cv_robusta.csv' -- ESTA es la cifra a citar en la memoria.


Obtención de predicciones 

In [10]:
# B) FIGURAS: matriz de confusión y curva ROC con predicciones
#    out-of-fold (una única partición 10-fold, sin repetición)

cv_visual = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEMILLA)

print("\n" + "=" * 70)
print("B) PREDICCIONES (10-fold único, N=85)")
print("=" * 70)

probas_oof_por_modelo = {}

for nombre, pipe in pipelines.items():
    probas_oof = cross_val_predict(pipe, X, y, cv=cv_visual, method='predict_proba')[:, 1]
    probas_oof_por_modelo[nombre] = probas_oof
    preds_oof = (probas_oof >= 0.5).astype(int)

    auc_oof = roc_auc_score(y, probas_oof)
    sens_oof = recall_score(y, preds_oof)
    cm = confusion_matrix(y, preds_oof)
    tn, fp, fn, tp = cm.ravel()
    esp_oof = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"\n--- {nombre} (OOF) ---")
    print(f"AUC-ROC: {auc_oof:.3f} | Sens: {sens_oof:.3f} | Esp: {esp_oof:.3f}")
    print(f"Matriz CM: TN={tn}, FP={fp}, FN={fn}, TP={tp}")


B) PREDICCIONES (10-fold único, N=85)

--- XGBoost (OOF) ---
AUC-ROC: 0.801 | Sens: 0.733 | Esp: 0.800
Matriz CM: TN=44, FP=11, FN=8, TP=22

--- Regresión Logística (OOF) ---
AUC-ROC: 0.866 | Sens: 0.867 | Esp: 0.745
Matriz CM: TN=41, FP=14, FN=4, TP=26

--- Random Forest (OOF) ---
AUC-ROC: 0.837 | Sens: 0.633 | Esp: 0.855
Matriz CM: TN=47, FP=8, FN=11, TP=19

--- TabPFN (OOF) ---
AUC-ROC: 0.852 | Sens: 0.667 | Esp: 0.818
Matriz CM: TN=45, FP=10, FN=10, TP=20


Generación y guardado de Curvas ROC y Matrices de Confusión

In [11]:
n_modelos = len(pipelines)
fig_roc, ax_roc = plt.subplots(figsize=(7, 6))
fig_cm, axes_cm = plt.subplots(1, n_modelos, figsize=(5 * n_modelos, 5))
if n_modelos == 1:
    axes_cm = [axes_cm]

for idx, (nombre, probas_oof) in enumerate(probas_oof_por_modelo.items()):
    preds_oof = (probas_oof >= 0.5).astype(int)
    auc_oof = roc_auc_score(y, probas_oof)
    cm = confusion_matrix(y, preds_oof)
    
    fpr, tpr, _ = roc_curve(y, probas_oof)
    ax_roc.plot(fpr, tpr, label=f"{nombre} (AUC = {auc_oof:.3f})")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['ypN0', 'ypN+'])
    disp.plot(ax=axes_cm[idx], colorbar=False, cmap='Blues')
    axes_cm[idx].set_title(nombre)

ax_roc.plot([0, 1], [0, 1], 'k--', label='Azar (AUC = 0.500)')
ax_roc.set_xlabel('1 - Especificidad')
ax_roc.set_ylabel('Sensibilidad')
ax_roc.set_title('Curva ROC — OOF (N=85)')
ax_roc.legend(loc='lower right')
fig_roc.tight_layout()
fig_roc.savefig(f"{CARPETA_FIGURAS}/curva_roc_out_of_fold.png", dpi=300)
plt.close(fig_roc)

fig_cm.tight_layout()
fig_cm.savefig(f"{CARPETA_FIGURAS}/matrices_confusion_out_of_fold.png", dpi=300)
plt.close(fig_cm)

print(f"\n[OK] Figuras guardadas en '{CARPETA_FIGURAS}/'")


[OK] Figuras guardadas en 'fig/'


3. Entrenamiento final sobre el dataset completo y guardado

In [12]:
for nombre, pipe in pipelines.items():
    pipe.fit(X, y)
    joblib.dump(pipe, f"{CARPETA_MODELOS}/pipeline_final_{nombre.lower().replace(' ', '_')}.pkl")

print(f"\n[OK] Modelos finales entrenados y guardados en '{CARPETA_MODELOS}/'")


[OK] Modelos finales entrenados y guardados en 'models/'
